# Configurando el ambiente

In [63]:
import sys

In [64]:
IS_COLAB = "google.colab" in sys.modules

In [65]:
if IS_COLAB:
    %pip install -q torchmetrics

In [66]:
import torch 
from sklearn.datasets import fetch_covtype
from sklearn.model_selection import train_test_split

In [67]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

'cuda'

# Importando el dataset y cargando los datos

In [68]:
covtype = fetch_covtype()

In [101]:
X_covtype = torch.tensor(covtype.data, dtype=torch.float32)
y_covtype = torch.tensor(covtype.target - 1, dtype=torch.long)

In [102]:
print(covtype.DESCR)

.. _covtype_dataset:

Forest covertypes
-----------------

The samples in this dataset correspond to 30×30m patches of forest in the US,
collected for the task of predicting each patch's cover type,
i.e. the dominant species of tree.
There are seven covertypes, making this a multiclass classification problem.
Each sample has 54 features, described on the
`dataset's homepage <https://archive.ics.uci.edu/ml/datasets/Covertype>`__.
Some of the features are boolean indicators,
while others are discrete or continuous measurements.

**Data Set Characteristics:**

=================   ============
Classes                        7
Samples total             581012
Dimensionality                54
Features                     int
=================   ============

:func:`sklearn.datasets.fetch_covtype` will load the covertype dataset;
it returns a dictionary-like 'Bunch' object
with the feature matrix in the ``data`` member
and the target values in ``target``. If optional argument 'as_frame' is
se

In [103]:
X_covtype.shape

torch.Size([581012, 54])

In [ ]:
means = X_covtype.mean(dim=0, keepdim=True)
std = X_covtype.std(dim=0, keepdim=True)

In [105]:
X_normalized = (X_covtype - means) / std

In [106]:
from torch.utils.data import TensorDataset

In [107]:
normalized_covtype = TensorDataset(X_normalized, y_covtype)

In [108]:
train_size = (len(X_normalized) * 70) // 100
valid_size = (len(X_normalized) * 70) // 100
test_size = len(X_normalized) - train_size - valid_size

In [109]:
from torch.utils.data import random_split

In [110]:
train_dataset, valid_dataset, test_dataset = random_split(normalized_covtype, lengths=[train_size, valid_size, test_size])

In [111]:
n_features = len(covtype.feature_names)
n_classes = len(set(covtype.target))

In [112]:
from torch.utils.data import DataLoader

batch_size = 256

In [113]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size)

# Definiendo nuestro modelo

In [114]:
import torch.nn as nn 
import torchmetrics
torch.manual_seed(21)

In [115]:
model = nn.Sequential(
    nn.Linear(n_features, 200), nn.ReLU(),
    nn.Linear(200, 200), nn.ReLU(),
    nn.Linear(200, 100), nn.ReLU(),
    nn.Linear(100, n_classes) # no función de activación ya que se utilizará softmax
).to(device)

In [116]:
def train_bgd(model, optimizer, criterion, metric, train_loader, n_epochs):
    model.train()
    metric.reset()
    for epoch in range(n_epochs):
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(preds, y_batch)
        mean_loss = total_loss / len(train_loader)
        metric_score = metric.compute().item()
        print(f"En época {epoch}:")
        print(f"Pérdida de: {mean_loss:.4f}")
        print(f"Precisión de: {metric_score:.2f}")

In [117]:
learning_rate = 0.1
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()
metric = torchmetrics.Accuracy(task='multiclass', num_classes=n_classes).to(device)

# Entrenando el modelo

In [118]:
train_bgd(model, optimizer, criterion, metric, train_loader, 100)

En época 0:
Pérdida de: 1.1130
Precisión de: 0.50
En época 1:
Pérdida de: 1.0487
Precisión de: 0.51
En época 2:
Pérdida de: 1.0218
Precisión de: 0.52
En época 3:
Pérdida de: 0.9919
Precisión de: 0.53
En época 4:
Pérdida de: 0.9604
Precisión de: 0.53
En época 5:
Pérdida de: 0.9295
Precisión de: 0.54
En época 6:
Pérdida de: 0.8964
Precisión de: 0.55
En época 7:
Pérdida de: 0.8723
Precisión de: 0.56
En época 8:
Pérdida de: 0.8440
Precisión de: 0.56
En época 9:
Pérdida de: 0.8204
Precisión de: 0.57
En época 10:
Pérdida de: 0.7985
Precisión de: 0.58
En época 11:
Pérdida de: 0.7792
Precisión de: 0.58
En época 12:
Pérdida de: 0.7593
Precisión de: 0.59
En época 13:
Pérdida de: 0.7432
Precisión de: 0.60
En época 14:
Pérdida de: 0.7263
Precisión de: 0.60
En época 15:
Pérdida de: 0.7137
Precisión de: 0.61
En época 16:
Pérdida de: 0.6996
Precisión de: 0.61
En época 17:
Pérdida de: 0.6865
Precisión de: 0.62
En época 18:
Pérdida de: 0.6743
Precisión de: 0.62
En época 19:
Pérdida de: 0.6629
Precisión

Ahora intentemoslo con regularización

In [119]:
learning_rate = 0.1
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
metric = torchmetrics.Accuracy(task='multiclass', num_classes=n_classes).to(device)

# Entrenando con regularización $l_2$

In [ ]:
model = nn.Sequential(
    nn.Linear(n_features, 200), nn.ReLU(),
    nn.Linear(200, 200), nn.ReLU(),
    nn.Linear(200, 100), nn.ReLU(),
    nn.Linear(100, n_classes) # no función de activación ya que se utilizará softmax
).to(device)

In [120]:
train_bgd(model, optimizer, criterion, metric, train_loader, 100)

En época 0:
Pérdida de: 0.3390
Precisión de: 0.86
En época 1:
Pérdida de: 0.3394
Precisión de: 0.86
En época 2:
Pérdida de: 0.3371
Precisión de: 0.86
En época 3:
Pérdida de: 0.3400
Precisión de: 0.86
En época 4:
Pérdida de: 0.3431
Precisión de: 0.86
En época 5:
Pérdida de: 0.3465
Precisión de: 0.86
En época 6:
Pérdida de: 0.3491
Precisión de: 0.86
En época 7:
Pérdida de: 0.3465
Precisión de: 0.86
En época 8:
Pérdida de: 0.3489
Precisión de: 0.86
En época 9:
Pérdida de: 0.3536
Precisión de: 0.86
En época 10:
Pérdida de: 0.3540
Precisión de: 0.86
En época 11:
Pérdida de: 0.3576
Precisión de: 0.85
En época 12:
Pérdida de: 0.3593
Precisión de: 0.85
En época 13:
Pérdida de: 0.3588
Precisión de: 0.85
En época 14:
Pérdida de: 0.3655
Precisión de: 0.85
En época 15:
Pérdida de: 0.3664
Precisión de: 0.85
En época 16:
Pérdida de: 0.3652
Precisión de: 0.85
En época 17:
Pérdida de: 0.3664
Precisión de: 0.85
En época 18:
Pérdida de: 0.3726
Precisión de: 0.85
En época 19:
Pérdida de: 0.3733
Precisión

Mejoró bastante nuestro modelo, intentemoslo ahora con early stopping

# Redefiniendo para incluir early stopping

In [125]:
def early_stopping_train_bgd(model, optimizer, criterion, metric, train_loader, valid_loader, n_epochs, patience_epochs=5):
    metric.reset()
    best_valid_loss = float("inf")
    epochs_without_improvement = 0
    for epoch in range(n_epochs):
        metric.reset()
        model.train()
        total_train_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            total_train_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(preds, y_batch)

        mean_train_loss = total_train_loss / len(train_loader)
        train_metric_score = metric.compute().item()    
        print(f"En época {epoch}:")
        print(f"Pérdida de: {mean_train_loss:.4f}")
        print(f"Precisión de: {train_metric_score:.2f}")

        model.eval()
        metric.reset()
        with torch.no_grad():
            total_valid_loss = 0
            for X_batch, y_batch in valid_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                preds = model(X_batch)
                loss = criterion(preds, y_batch)
                total_valid_loss += loss.item()
                metric.update(preds, y_batch)
            mean_valid_loss = total_valid_loss / len(valid_loader)
            valid_metric_score = metric.compute().item()
            print(f"Pérdida en conjunto de validación: {mean_valid_loss:.4f}")
            print(f"Precisión en conjunto de validación: {valid_metric_score:.2f}")
            if mean_valid_loss <= best_valid_loss:
                best_valid_loss = mean_valid_loss
                epochs_without_improvement = 0
            else: 
                epochs_without_improvement += 1
                if epochs_without_improvement == patience_epochs:
                    print("Pérdida en conjunto de validación aumentando, parando entrenamiento...")
                    print(f"Épocas de entrenamiento: {epoch}")
                    print(f"Pérdida de entrenamiento alcanzada: {mean_train_loss}")
                    print(f"Precisión en conjunto de entrenamiento: {train_metric_score}")
                    print(f"Pérdida de validación alcanzada: {mean_valid_loss}")
                    print(f"Precisión en conjunto de validación: {valid_metric_score}")
                    return # regresamos cuando la pérdida empiece a subir

Ahora intentemos nuestra prueba con regularización otra vez

In [127]:
model = nn.Sequential(
    nn.Linear(n_features, 200), nn.ReLU(),
    nn.Linear(200, 200), nn.ReLU(),
    nn.Linear(200, 100), nn.ReLU(),
    nn.Linear(100, n_classes) # no función de activación ya que se utilizará softmax
).to(device)

In [128]:
early_stopping_train_bgd(model, optimizer, criterion, metric, train_loader, valid_loader, 100)

En época 0:
Pérdida de: 1.9484
Precisión de: 0.27
Pérdida en conjunto de validación: 1.9486
Precisión en conjunto de validación: 0.27
En época 1:
Pérdida de: 1.9484
Precisión de: 0.27
Pérdida en conjunto de validación: 1.9486
Precisión en conjunto de validación: 0.27
En época 2:
Pérdida de: 1.9484
Precisión de: 0.27
Pérdida en conjunto de validación: 1.9486
Precisión en conjunto de validación: 0.27
En época 3:
Pérdida de: 1.9484
Precisión de: 0.27
Pérdida en conjunto de validación: 1.9486
Precisión en conjunto de validación: 0.27
En época 4:
Pérdida de: 1.9484
Precisión de: 0.27
Pérdida en conjunto de validación: 1.9486
Precisión en conjunto de validación: 0.27
En época 5:
Pérdida de: 1.9484
Precisión de: 0.27
Pérdida en conjunto de validación: 1.9486
Precisión en conjunto de validación: 0.27
En época 6:
Pérdida de: 1.9484
Precisión de: 0.27
Pérdida en conjunto de validación: 1.9486
Precisión en conjunto de validación: 0.27
En época 7:
Pérdida de: 1.9484
Precisión de: 0.27
Pérdida en c

KeyboardInterrupt: 